In [2]:
#!/usr/bin/env python3
import sys
import warnings
from Bio.PDB import MMCIFParser, PDBIO
from Bio.PDB.PDBExceptions import PDBConstructionWarning

def convert_cif_to_pdb(input_cif, output_pdb):
    # 忽略 PDB 构建过程中的常规警告
    warnings.simplefilter("ignore", PDBConstructionWarning)
    
    # 解析 CIF 文件
    parser = MMCIFParser()
    print(f"正在读取 CIF 文件: {input_cif}")
    try:
        structure = parser.get_structure("protein", input_cif)
    except Exception as e:
        print(f"解析失败: {e}")
        sys.exit(1)
        
    # 保存为 PDB 文件
    io = PDBIO()
    io.set_structure(structure)
    try:
        io.save(output_pdb)
        print(f"转换成功，已保存至: {output_pdb}")
    except Exception as e:
        print(f"保存失败: {e}")
        sys.exit(1)

if __name__ == "__main__":
    # 支持命令行传参：python cif2pdb.py <input.cif> <output.pdb>
    # 如果没有传参，则默认使用你指定的路径
    if len(sys.argv) == 3:
        input_file = sys.argv[1]
        output_file = sys.argv[2]
    else:
        input_file = "/share/home/wangtb/enzyme_shells/structure/PTEN.cif"
        output_file = "/share/home/wangtb/enzyme_shells/structure/PTEN.pdb"
        
    convert_cif_to_pdb(input_file, output_file)

正在读取 CIF 文件: /share/home/wangtb/enzyme_shells/structure/PTEN.cif
转换成功，已保存至: /share/home/wangtb/enzyme_shells/structure/PTEN.pdb


In [ ]:
import pandas as pd
import os
import math
import glob

# ================= 路径配置 =================
csv_dir = "/share/home/wangtb/enzyme_shells/activity_data"
pdb_dir = "/share/home/wangtb/enzyme_shells/structure"
output_base_dir = "/share/home/wangtb/enzyme_shells/foldx"
mutants_base_dir = os.path.join(output_base_dir, "mutants")
foldx_exe = "/share/home/wangtb/foldx/foldx_20270131" # 依据你的环境确认此路径

chunk_size = 50
target_chain = "A"

# 获取所有的 csv 文件
csv_files = glob.glob(os.path.join(csv_dir, "*.csv"))

for csv_file in csv_files:
    # 提取无后缀的纯文件名作为 dataset_id (如 AICDA, AMIE)
    filename = os.path.basename(csv_file)
    dataset_id = filename.replace('.csv', '')
    
    # 根据需求：结构命名和 csv 命名一致
    pdb_name = f"{dataset_id}.pdb"
    pdb_file = os.path.join(pdb_dir, pdb_name)
    
    # 检查输入文件是否存在
    if not os.path.exists(pdb_file):
        print(f"⚠️ 找不到对应的 PDB 文件: {pdb_file}，跳过 {dataset_id} ...")
        continue

    # 创建工作目录和存放突变的专属目录
    work_dir = os.path.join(output_base_dir, dataset_id)
    split_dir = os.path.join(mutants_base_dir, dataset_id)
    os.makedirs(work_dir, exist_ok=True)
    os.makedirs(split_dir, exist_ok=True)
    
    # 读取数据并提取突变
    df = pd.read_csv(csv_file)
    if 'mutant' not in df.columns:
        print(f"⚠️ {dataset_id} 中不存在 mutant 列，跳过...")
        continue
        
    mutations = []
    for mut in df['mutant'].dropna():
        if mut != 'WT':
            wt_aa = mut[0]
            mut_aa = mut[-1]
            pos = mut[1:-1]
            # 拼接成 FoldX 格式: QA21G;
            foldx_mut = f"{wt_aa}{target_chain}{pos}{mut_aa};"
            mutations.append(foldx_mut)

    # 如果没有有效突变，跳过
    if not mutations:
        print(f"⚠️ {dataset_id} 中没有提取到有效突变，跳过...")
        continue

    # 拆分列表写入独立的 mutants 文件夹中
    num_chunks = math.ceil(len(mutations) / chunk_size)
    os.system(f"rm -f {split_dir}/mut_list_*.txt") # 清理旧文件
    
    for i in range(1, num_chunks + 1):
        chunk_lines = mutations[(i-1)*chunk_size : i*chunk_size]
        with open(os.path.join(split_dir, f"mut_list_{i}.txt"), 'w') as f:
            f.write("\n".join(chunk_lines) + "\n")

    # 动态生成专属的 Slurm 提交脚本
    slurm_script_path = os.path.join(work_dir, "run_foldx.sh")
    slurm_content = f"""#!/bin/bash
#SBATCH --job-name=FX_{dataset_id[:10]}
#SBATCH --partition=cu-1
#SBATCH --array=1-{num_chunks}
#SBATCH --output=logs/foldx_%A_%a.out
#SBATCH --error=logs/foldx_%A_%a.err

# 切换到当前蛋白的工作目录
cd {work_dir}

mkdir -p logs
mkdir -p results_all

TASK_ID=$SLURM_ARRAY_TASK_ID
# 从新建立的 mutants 文件夹读取突变列表
MUT_FILE="{split_dir}/mut_list_${{TASK_ID}}.txt"

if [ ! -f "$MUT_FILE" ]; then
    echo "错误：找不到文件 $MUT_FILE"
    exit 1
fi

TEMP_DIR="temp_task_${{TASK_ID}}"
mkdir -p ${{TEMP_DIR}}

# 拷贝 PDB 和当前突变列表到临时目录
cp {pdb_file} ${{TEMP_DIR}}/
cp ${{MUT_FILE}} ${{TEMP_DIR}}/individual_list.txt

cd ${{TEMP_DIR}}

{foldx_exe} --command=BuildModel \\
      --pdb={pdb_name} \\
      --pdb-dir=./ \\
      --mutant-file=individual_list.txt \\
      --numberOfRuns=1 \\
      --output-dir=./

# 提取并重命名结果文件 
DIF_FILE="Dif_{dataset_id}.fxout"

if [ -f "$DIF_FILE" ]; then
    cp $DIF_FILE ../results_all/Dif_result_${{TASK_ID}}.fxout
    echo "任务 ${{TASK_ID}} 结果提取成功。"
else
    echo "警告：任务 ${{TASK_ID}} 未生成 $DIF_FILE"
fi

cd ..
rm -rf ${{TEMP_DIR}}
"""
    with open(slurm_script_path, 'w') as f:
        f.write(slurm_content)

    print(f"[成功] {dataset_id}: 生成 {num_chunks} 个任务，PDB={pdb_name}，突变文件存放于 {split_dir}")

print("\n🎉 所有任务编排完成！")